# Bảng ứng viên PubChem (retrieval) cho massbank / msnlib trên Kaggle

Chạy `data_scripts/public/make_pubchem_candidates.py` (nhánh `claude/code-explanation-frix3g`): với mỗi phổ của tập test,
lấy phân tử đúng + 49 đồng phân PubChem cùng công thức giống nhất (Tanimoto), như `data_scripts/pubchem/04_make_retrieval_lists.py`
của tác giả. Kết quả: `retrieval/cands_df_split_1_50.tsv` cho từng tập.

**Đầu vào:** *Add Input* output của notebook dựng dữ liệu (`kaggle_build_public_data.ipynb`) cho massbank và/hoặc msnlib.
Notebook chỉ cần `labels.tsv` và `splits/` của chúng.

| `STAGE` | Làm gì | Môi trường |
|---|---|---|
| `"scan"` | Tải `CID-SMILES.gz` từ NCBI, quét phần `SHARD_ID` / `NUM_SHARDS` của PubChem, giữ phân tử trùng công thức | chỉ cần rdkit (cài nhanh) |
| `"build"` | Gộp các shard (*Add Input* output các notebook scan), chọn 50 ứng viên, ghi bảng | môi trường ms-pred (~10 phút) |
| `"all"` | Cả hai trong một phiên | môi trường ms-pred |

**Cách chạy:** *Accelerator: None*, *Internet: On*.
1. Thử `STAGE = "all"` trước (*Save Version → Save & Run All*). Log in tiến độ quét kèm số phút. Nếu ước lượng quá 11 giờ,
   dừng lại và chia shard.
2. Chia shard: `NUM_SHARDS` bản sao `STAGE = "scan"`, `SHARD_ID = 0 … NUM_SHARDS-1` (chạy song song), rồi một notebook
   `STAGE = "build"` với *Add Input* output của mọi notebook scan (và output dữ liệu massbank/msnlib).
3. Output: `ms-pred/data/spec_datasets/<ds>/retrieval/cands_df_split_1_50.tsv`. Chép vào `data/spec_datasets/<ds>/retrieval/`
   trên máy train; `run_marason_baseline.py` sẽ tự chạy retrieval (B6) cho tập đó.

In [ ]:
# ===== CẤU HÌNH =====
DATASETS     = ["massbank", "msnlib"]   # tập cần bảng ứng viên (phải có trong Input)
STAGE        = "all"                    # "scan" | "build" | "all"
NUM_SHARDS   = 1                        # giống nhau ở mọi notebook scan và build
SHARD_ID     = 0                        # STAGE = "scan": 0 .. NUM_SHARDS-1
SUBSETS      = ["test"]                 # thêm "val" nếu cần bảng cho tập val
INPUTS       = ["/kaggle/input"]        # nơi tìm dữ liệu và output các notebook scan (tìm đệ quy)

REPO_URL     = "https://github.com/Giamyy/ms-pred.git"
REPO_BRANCH  = "claude/code-explanation-frix3g"   # nhánh có make_pubchem_candidates.py

In [ ]:
import os, subprocess, sys, json, shutil, time
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = Path("/tmp/ms-pred")
NCPU = os.cpu_count() or 4
PUBCHEM = Path("/tmp/pubchem/CID-SMILES.gz")
CAND_DIR = REPO / "data/retrieval/pubchem/public_candidates"
assert STAGE in {"scan", "build", "all"}, STAGE
assert 0 <= SHARD_ID < NUM_SHARDS

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh shell, in output ngay khi có; bấm Interrupt sẽ dừng cả tiến trình con."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", start_new_session=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    p.tail = []
    try:
        for line in p.stdout:
            p.tail = (p.tail + [line])[-200:]
            if "it/s]" in line or "s/it]" in line:
                continue
            print(line, end="")
    except KeyboardInterrupt:
        os.killpg(p.pid, 9)
        raise
    if p.wait() != 0 and check:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): {cmd}")
    return p

import urllib.request
try:
    urllib.request.urlopen("https://ftp.ncbi.nlm.nih.gov/pubchem/", timeout=20)
    print("Internet OK")
except Exception as e:
    raise SystemExit(f"Chưa có internet (hoặc không vào được ftp.ncbi.nlm.nih.gov): Settings -> Internet -> On.\n{e}")
print("CPU cores:", NCPU, "| STAGE:", STAGE, "| DATASETS:", DATASETS)

In [ ]:
# ===== Lấy code, chép labels.tsv + splits của các tập từ Input =====
if not REPO.exists():
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO}")
for ds in DATASETS:
    cands = sorted({p.parent for root in INPUTS for p in Path(root).glob(f"**/spec_datasets/{ds}/labels.tsv")},
                   key=lambda d: len(d.parts))
    if not cands:
        raise SystemExit(f"Không thấy spec_datasets/{ds}/labels.tsv trong {INPUTS}: hãy Add Input output dữ liệu {ds}")
    src, dst = cands[0], REPO / "data/spec_datasets" / ds
    (dst / "splits").mkdir(parents=True, exist_ok=True)
    shutil.copy2(src / "labels.tsv", dst / "labels.tsv")
    for f in (src / "splits").glob("*.tsv"):
        shutil.copy2(f, dst / "splits" / f.name)
    print(f"{ds}: {src}")

Ô dưới ghi đoạn cài môi trường ms-pred (giống notebook dựng dữ liệu, có cách dự phòng khi không tải được DGL) ra file;
ô tiếp theo chỉ chạy nó khi cần (`STAGE` build / all).

In [ ]:
%%writefile /tmp/install_env.py
# ===== 1. Clone repo + cài môi trường CPU (~10 phút) =====
if not REPO.exists():
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO}")
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
PY = str(REPO / ".venv" / "bin" / "python")
CHECK = "import sys, rdkit, h5py, ms_pred; from ms_pred.magma import fragmentation"
def env_ready():
    # .venv có thể còn sót lại (rỗng hoặc thiếu gói) từ một lần uv sync lỗi trước: kiểm tra bằng import thật
    return Path(PY).exists() and subprocess.run([PY, "-c", CHECK], capture_output=True).returncode == 0
def lite_pyproject(text, keep_pytorch_index=True):
    """pyproject.toml bỏ dgl / torch-scatter / torch-sparse và nguồn wheel của chúng (data.dgl.ai, data.pyg.org).

    Các bước dựng dữ liệu (chuyển đổi, MAGMa, subformula) không gọi tới các gói này: ms_pred chỉ import chúng,
    nên module giả (stub) là đủ. keep_pytorch_index=False bỏ luôn download.pytorch.org (torch lấy từ PyPI).
    """
    out, section = [], None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("["):
            section = s.strip("[]")
        if section in ("tool.uv", "tool.uv.sources", "tool.uv.index"):
            continue
        if s.startswith("cpu = ["):
            line = 'cpu = ["torch==2.6.0", "torchvision==0.21.0", "torch-geometric"]'
        elif s.startswith(("cu124 = [", "cu118 = [")):
            continue
        out.append(line)
    if keep_pytorch_index:
        out += ["", "[tool.uv.sources]",
                'torch = { index = "pytorch-cpu" }', 'torchvision = { index = "pytorch-cpu" }',
                "", "[[tool.uv.index]]", 'name = "pytorch-cpu"', 'url = "https://download.pytorch.org/whl/cpu"',
                "explicit = true"]
    return "\n".join(out) + "\n"

STUB = """# Module giả: chỉ để `import` được khi dựng dữ liệu (không gọi tới thư viện thật)
import sys, types
class _Any(types.ModuleType):
    def __getattr__(self, name):
        if name.startswith("__"):
            raise AttributeError(name)
        return _Any(f"{self.__name__}.{name}")
    def __call__(self, *a, **k):
        return _Any("call")
    def __mro_entries__(self, bases):
        return (object,)
for _sub in ("function", "nn", "nn.pytorch", "nn.pytorch.glob", "backend", "ops", "data", "utils", "readout", "sparse"):
    sys.modules[f"{__name__}.{_sub}"] = _Any(f"{__name__}.{_sub}")
def __getattr__(name):
    return _Any(f"{__name__}.{name}")
__version__ = "stub"
"""

def write_stubs():
    site = subprocess.run([PY, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                          capture_output=True, text=True).stdout.strip()
    for mod in ("dgl", "torch_scatter", "torch_sparse"):
        if subprocess.run([PY, "-c", f"import {mod}"], capture_output=True).returncode != 0:
            (Path(site) / mod).mkdir(exist_ok=True)
            (Path(site) / mod / "__init__.py").write_text(STUB)
            print(f"Dùng module giả cho {mod} (không cần khi dựng dữ liệu)")

if not env_ready():
    # Giống hệt notebook dựng MassSpecGym (kaggle_build_msg_data.ipynb): không nâng cấp uv có sẵn trên Kaggle
    sh("pip install -q uv && uv --version")
    pyproject = REPO / "pyproject.toml"
    original = pyproject.read_text()
    plans = [("pyproject.toml gốc, như notebook MassSpecGym", None),
             ("pyproject.toml gốc, thử lại", None),
             ("bỏ dgl / torch-scatter / torch-sparse (data.dgl.ai, data.pyg.org)", True),
             ("bỏ thêm download.pytorch.org, torch lấy từ PyPI", False)]
    for attempt, (label, keep_pytorch) in enumerate(plans, 1):
        pyproject.write_text(original if keep_pytorch is None else lite_pyproject(original, keep_pytorch))
        print(f"\n===== Cài lần {attempt}/{len(plans)}: {label} =====")
        # --python 3.11: Python 3.13 của Kaggle không có wheel torch-scatter / torch-sparse dựng sẵn
        p = sh("uv sync --extra cpu --python 3.11", cwd=REPO, check=False)   # cùng lệnh với notebook MassSpecGym
        if p.returncode == 0:
            break
        errs = [l.rstrip() for l in p.tail if any(w in l.lower() for w in ("error", "failed", "caused by", "cause:", "hint", "×", "╰─"))]
        print(f"\n===== uv sync lỗi (lần {attempt}/{len(plans)}). Các dòng lỗi: =====")
        print("\n".join(errs[-25:]) or "".join(p.tail[-25:]))
        time.sleep(10)
    else:
        raise RuntimeError("uv sync thất bại mọi cách: xem các dòng lỗi ở trên (gửi ảnh chụp phần đó để sửa)")
    write_stubs()
sh(f"{PY} -c \"{CHECK}; print('python', sys.version.split()[0], '| rdkit', rdkit.__version__)\"")
# Các script của tác giả gọi `python`/`python3`: cho chúng dùng môi trường vừa cài
os.environ["PATH"] = f"{REPO / '.venv' / 'bin'}:{os.environ['PATH']}"

In [ ]:
# ===== Môi trường =====
if STAGE == "scan":
    # Quét chỉ cần rdkit + pandas: cài thẳng vào Python của Kaggle (nhanh), cùng phiên bản rdkit với pyproject.toml
    sh('pip install -q "rdkit==2025.3.*"')
    PY = sys.executable
else:
    exec(open("/tmp/install_env.py").read())      # đặt PY = /tmp/ms-pred/.venv/bin/python

In [ ]:
# ===== Quét PubChem / dựng bảng =====
args = f"--datasets {' '.join(DATASETS)} --subsets {' '.join(SUBSETS)} --pubchem {PUBCHEM} --work-dir {CAND_DIR} " \
       f"--num-shards {NUM_SHARDS} --workers {NCPU}"
t0 = time.time()
if STAGE in ("scan", "all"):
    sh(f"{PY} data_scripts/public/make_pubchem_candidates.py {args} --stage scan --shard-id {SHARD_ID} --download", cwd=REPO)
if STAGE == "build":
    # lấy kết quả quét của các notebook scan từ Input
    CAND_DIR.mkdir(parents=True, exist_ok=True)
    found = {}
    for root in INPUTS:
        for p in Path(root).glob(f"**/matches_shard*of{NUM_SHARDS}.done.json"):
            found[p.name] = p
    for name, done in sorted(found.items()):
        shutil.copy2(done, CAND_DIR / name)
        shutil.copy2(done.parent / name.replace(".done.json", ".tsv"), CAND_DIR / name.replace(".done.json", ".tsv"))
        print("Tìm thấy:", done.parent, json.loads(done.read_text()))
    if len(found) != NUM_SHARDS:
        raise RuntimeError(f"Cần đủ {NUM_SHARDS} shard, tìm thấy {sorted(found)}")
if STAGE in ("build", "all"):
    sh(f"{PY} data_scripts/public/make_pubchem_candidates.py {args} --stage build", cwd=REPO)
print(f"Xong sau {(time.time() - t0) / 60:.0f} phút")

In [ ]:
# ===== Xuất ra /kaggle/working =====
if STAGE == "scan":
    out = WORK / "pubchem_candidates"
    out.mkdir(exist_ok=True)
    for f in CAND_DIR.glob(f"matches_shard{SHARD_ID}of{NUM_SHARDS}*"):
        shutil.copy2(f, out / f.name)
    shutil.copy2(CAND_DIR / "targets.json", out / f"targets_shard{SHARD_ID}.json")
    sh(f"ls -lh {out}")
    print("Save Version -> Save & Run All; khi đủ shard thì chạy notebook STAGE = 'build'.")
else:
    for ds in DATASETS:
        src = REPO / "data/spec_datasets" / ds / "retrieval"
        dst = WORK / "ms-pred/data/spec_datasets" / ds / "retrieval"
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.glob("cands_df_*.tsv"):
            shutil.copy2(f, dst / f.name)
        sh(f"ls -lh {dst}")
    shutil.copy2(CAND_DIR / "build_summary.json", WORK / "build_summary.json")
    print((WORK / "build_summary.json").read_text())